# Orientation analysis

This notebook builds a small synthetic rotation sweep and computes zone-axis and excitation-density descriptors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact, IntSlider

from src.geometry import UnitCell, cell_to_reciprocal_matrix, rotation_matrix_from_axis_angle
from src.orientation_metrics import OrientationMetricConfig, compute_orientation_metrics
from src.visualization import plot_orientation_trajectory, plot_zone_axis_distance

In [ ]:
cell = UnitCell(a=18.5, b=18.5, c=27.2, alpha=90, beta=90, gamma=120)
ub0 = cell_to_reciprocal_matrix(cell)
rows = []
for frame in range(1, 31):
    rot = rotation_matrix_from_axis_angle([0, 1, 0], (frame - 1) * 1.5)
    ub = rot @ ub0
    rows.append({
        'frame': frame,
        'phi': (frame - 1) * 1.5,
        'UB11': ub[0,0], 'UB12': ub[0,1], 'UB13': ub[0,2],
        'UB21': ub[1,0], 'UB22': ub[1,1], 'UB23': ub[1,2],
        'UB31': ub[2,0], 'UB32': ub[2,1], 'UB33': ub[2,2],
    })
orientations = pd.DataFrame(rows)
summary = compute_orientation_metrics(orientations, cell, OrientationMetricConfig(dmin=0.8, dmax=15.0, hkl_limit=12))
summary.head()

In [ ]:
plot_orientation_trajectory(summary);
plot_zone_axis_distance(summary);
plt.show()

In [ ]:
@interact(frame=IntSlider(min=1, max=int(summary['frame'].max()), step=1, value=1))
def inspect_frame(frame=1):
    display(summary[summary['frame'] == frame])